In [50]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

# Your client credentials
client_id = 'sh-ebcb9744-76f7-4d5b-b0c8-620009ececcd'
client_secret = 'GiZXQ53eTweIdIdIcq1l3to1rUVgEqes'

# Create a session
client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

# Get token for the session
token = oauth.fetch_token(token_url='https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
                          client_secret=client_secret, include_client_id=True)

# All requests using this session will have an access token automatically added
resp = oauth.get("https://sh.dataspace.copernicus.eu/configuration/v1/wms/instances")
print(resp.content)

b'[]'


In [51]:
from sentinelhub import SHConfig

config = SHConfig()
config.sh_client_id = 'sh-ebcb9744-76f7-4d5b-b0c8-620009ececcd'
config.sh_client_secret = 'GiZXQ53eTweIdIdIcq1l3to1rUVgEqes'
config.sh_base_url = 'https://sh.dataspace.copernicus.eu'
config.sh_token_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
config.save("my-profile")

if not config.sh_client_id or not config.sh_client_secret:
    print("Warning! To use Process API, please provide the credentials (OAuth client ID and client secret).")

In [52]:
%reload_ext autoreload
%autoreload 2
# %matplotlib inline

In [53]:
import datetime
import os

# import matplotlib.pyplot as plt
import numpy as np

from sentinelhub import (
    CRS,
    BBox,
    DataCollection,
    DownloadRequest,
    MimeType,
    MosaickingOrder,
    SentinelHubDownloadClient,
    SentinelHubRequest,
    bbox_to_dimensions,
)

# The following is not a package. It is a file utils.py which should be in the same folder as this notebook.
# from utils import plot_image

In [54]:
airth_coords_wgs84 = (56.143233900690035, -3.9261835798698144,
                      56.14950099836631, -3.910343307218369)

In [55]:
resolution = 10
airth_bbox = BBox(bbox=airth_coords_wgs84, crs=CRS.WGS84)
airth_size = bbox_to_dimensions(airth_bbox, resolution=resolution)

print(f"Image shape at {resolution} m resolution: {airth_size} pixels")

Image shape at 10 m resolution: (69, 175) pixels


In [56]:
evalscript_all_bands = """
    //VERSION=3
    function setup() {
        return {
            input: [{
                bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B10","B11","B12"],
                units: "DN"
            }],
            output: {
                bands: 13,
                sampleType: "INT16"
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B01,
                sample.B02,
                sample.B03,
                sample.B04,
                sample.B05,
                sample.B06,
                sample.B07,
                sample.B08,
                sample.B8A,
                sample.B09,
                sample.B10,
                sample.B11,
                sample.B12];
    }
"""

request_all_bands = SentinelHubRequest(
    evalscript=evalscript_all_bands,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L1C,
            time_interval=("2020-06-01", "2020-06-30"),
            mosaicking_order=MosaickingOrder.LEAST_CC,
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=airth_bbox,
    size=airth_size,
    config=config,
)

In [57]:
config

SHConfig(
  instance_id='',
  sh_client_id='***********************************eccd',
  sh_client_secret='****************************Eqes',
  sh_base_url='https://sh.dataspace.copernicus.eu',
  sh_auth_base_url=None,
  sh_token_url='https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
  geopedia_wms_url='https://service.geopedia.world',
  geopedia_rest_url='https://www.geopedia.world/rest',
  aws_access_key_id='',
  aws_secret_access_key='',
  aws_session_token='',
  aws_metadata_url='https://roda.sentinel-hub.com',
  aws_s3_l1c_bucket='sentinel-s2-l1c',
  aws_s3_l2a_bucket='sentinel-s2-l2a',
  opensearch_url='http://opensearch.sentinel-hub.com/resto/api/collections/Sentinel2',
  max_wfs_records_per_query=100,
  max_opensearch_records_per_query=500,
  max_download_attempts=4,
  download_sleep_time=5.0,
  download_timeout_seconds=120.0,
  number_of_download_processes=1,
)

In [58]:
all_bands_response = request_all_bands.get_data()

DownloadFailedException: Failed to download from:
https://services.sentinel-hub.com/api/v1/process
with HTTPError:
401 Client Error: Unauthorized for url: https://services.sentinel-hub.com/api/v1/process
Server response: "{"status": 401, "reason": "Unauthorized", "message": "You are not authorized! Please provide a valid access token within the header [Authorization: Bearer <accessToken>] of your request.", "code": "COMMON_UNAUTHORIZED"}"

In [6]:
# search by polygon, time, and SciHub query keywords
footprint = geojson_to_wkt({
                              "type": "FeatureCollection",
                              "features": [
                                {
                                  "type": "Feature",
                                  "properties": {},
                                  "geometry": {
                                    "coordinates": [
                                      [
                                        [
                                          -3.9261835798698144,
                                          56.14950099836631
                                        ],
                                        [
                                          -3.9261835798698144,
                                          56.143233900690035
                                        ],
                                        [
                                          -3.910343307218369,
                                          56.143233900690035
                                        ],
                                        [
                                          -3.910343307218369,
                                          56.14950099836631
                                        ],
                                        [
                                          -3.9261835798698144,
                                          56.14950099836631
                                        ]
                                      ]
                                    ],
                                    "type": "Polygon"
                                  }
                                }
                              ]
                            }
                                )
products = api.query(footprint,
                     date=('20151219', date(2015, 12, 29)),
                     platformname='Sentinel-2',
                     cloudcoverpercentage=(0, 30))

ConnectTimeout: HTTPSConnectionPool(host='apihub.copernicus.eu', port=443): Max retries exceeded with url: /apihub/search?format=json&rows=100&start=0&q=beginPosition%3A%5B%222015-12-19T00%3A00%3A00Z%22+TO+%222015-12-29T00%3A00%3A00Z%22%5D+cloudcoverpercentage%3A%5B%220%22+TO+%2230%22%5D+platformname%3A%22Sentinel-2%22+footprint%3A%22Intersects%28GEOMETRYCOLLECTION%28POLYGON%28%28-3.9262+56.1495%2C-3.9262+56.1432%2C-3.9103+56.1432%2C-3.9103+56.1495%2C-3.9262+56.1495%29%29%29%29%22 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x10549fd40>, 'Connection to apihub.copernicus.eu timed out. (connect timeout=None)'))

In [5]:
footprint

'GEOMETRYCOLLECTION(POLYGON((-3.9262 56.1495,-3.9262 56.1432,-3.9103 56.1432,-3.9103 56.1495,-3.9262 56.1495)))'

In [59]:
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04"],
    mosaicking: Mosaicking.ORBIT,
    output: { id: "default", bands: 3 },
  }
}

function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
  outputMetadata.userData = { scenes: scenes.orbits }
}

function evaluatePixel(samples) {
  return [2.5 * samples[0].B04, 2.5 * samples[0].B03, 2.5 * samples[0].B02]
}
"""

request = {
    "input": {
        "bounds": {
            "bbox": [
                13.822174072265625,
                45.85080395917834,
                14.55963134765625,
                46.29191774991382,
            ]
        },
        "data": [
            {
                "type": "sentinel-2-l1c",
                "dataFilter": {
                    "timeRange": {
                        "from": "2018-12-27T00:00:00Z",
                        "to": "2018-12-27T23:59:59Z",
                    }
                },
            }
        ],
    },
    "output": {
        "width": 512,
        "height": 512,
        "responses": [
            {
                "identifier": "default",
                "format": {"type": "image/tiff"},
            },
            {
                "identifier": "userdata",
                "format": {"type": "application/json"},
            },
        ],
    },
    "evalscript": evalscript,
}

url = "https://sh.dataspace.copernicus.eu/api/v1/process"
response = oauth.post(url, json=request, headers={"Accept": "application/tar"})

In [60]:
response

<Response [200]>